# MNIST CNN — Augmented Training for STM32N6570-DK

Same architecture as the original notebook (`digitsMINST.ipynb`):
- Input: **32×32×3 float32** — firmware does not change
- Output: **10-class softmax** — firmware does not change

What is different: training uses **data augmentation** so the model learns to handle
real handwriting variation (tilt, off-center position, different sizes, stroke style).

After running all cells, download `mnist_cnn_32x32.onnx` and convert it with STEdgeAI.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

## 1 — Load and preprocess MNIST

Same preprocessing as the original notebook:
- resize 28×28 → 32×32 to match the firmware input
- repeat grayscale to 3 channels (firmware expects RGB)
- normalize to [0, 1] — MNIST convention: **0 = background, 1 = digit stroke**

This matches what the firmware preprocessing produces (inverted grayscale makes
white paper → 0, dark ink → 1).

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

def preprocess(images):
    images = images[..., np.newaxis].astype('float32') / 255.0  # [N,28,28,1], range [0,1]
    images = tf.image.resize(images, [32, 32]).numpy()           # [N,32,32,1]
    images = np.repeat(images, 3, axis=-1)                       # [N,32,32,3]
    return images

x_train = preprocess(x_train)
x_test  = preprocess(x_test)
print('Train:', x_train.shape, '| Test:', x_test.shape)

## 2 — Data augmentation

Each parameter targets a specific real-world problem:

| Parameter | Value | Why |
|-----------|-------|-----|
| `rotation_range` | 15° | Digits written at a slight angle |
| `width_shift_range` | 10% | Digit not perfectly centred horizontally |
| `height_shift_range` | 10% | Digit not perfectly centred vertically |
| `zoom_range` | 15% | Digit held closer or farther from camera |
| `shear_range` | 10° | Slanted handwriting |
| `fill_mode='constant', cval=0` | — | Empty border filled with black (background), not a copy of the edge |

Augmentation is applied **only during training**, not at inference time.
The exported model is identical in structure to the original.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    shear_range=10,
    fill_mode='constant',
    cval=0.0
)

# Fit on a subset (computes internal statistics if needed)
datagen.fit(x_train[:1000])

### Preview — what augmented digits look like

Run this cell to visualise a few examples before training.
Each row shows the original digit and 7 augmented versions.

In [ ]:
fig, axes = plt.subplots(5, 8, figsize=(14, 9))
fig.suptitle('Left: original MNIST   |   Right: augmented versions', fontsize=12)

# Pick one example per digit class 0-4
for row, digit in enumerate(range(5)):
    idx = np.where(y_train == digit)[0][0]
    sample = x_train[idx:idx+1]          # shape (1,32,32,3)

    axes[row, 0].imshow(sample[0, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(f'digit {digit}', fontsize=8)
    axes[row, 0].axis('off')

    gen = datagen.flow(sample, batch_size=1)
    for col in range(1, 8):
        aug = next(gen)[0]
        axes[row, col].imshow(aug[:, :, 0], cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 3 — Model

**Identical architecture to the original notebook.**
Input/output shape unchanged → firmware needs no modifications.

In [ ]:
inputs  = tf.keras.Input(shape=(32, 32, 3), name='input')
x = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')(inputs)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
outputs = tf.keras.layers.Dense(10, activation='softmax', name='output')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

## 4 — Training with augmentation

More epochs than the original (20 vs 10): augmentation makes each epoch harder
because the model never sees the same image twice, so it needs more passes to converge.

Expected test accuracy on clean MNIST: **~97–98%** (similar to original).
Real-world performance on handwritten digits will be noticeably better.

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Split off 10% of training data for validation (same as original)
val_split = int(len(x_train) * 0.1)
x_val, y_val = x_train[:val_split], y_train[:val_split]
x_tr,  y_tr  = x_train[val_split:], y_train[val_split:]

BATCH_SIZE     = 128
EPOCHS         = 20
steps_per_epoch = len(x_tr) // BATCH_SIZE

history = model.fit(
    datagen.flow(x_tr, y_tr, batch_size=BATCH_SIZE),
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=(x_val, y_val),
    verbose=1
)

loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f'\nTest accuracy on clean MNIST: {acc:.4f}')

### Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy')
ax1.set_xlabel('epoch')
ax1.legend()

ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss')
ax2.set_xlabel('epoch')
ax2.legend()

plt.tight_layout()
plt.show()

## 5 — Export to ONNX

Produces `mnist_cnn_32x32.onnx` — same filename as the original so the STEdgeAI
conversion config does not need to change.

Also saves a calibration file (`mnist_calib.npz`) used by STEdgeAI for quantisation.

In [ ]:
!pip install tf2onnx -q

import tf2onnx

input_sig = [tf.TensorSpec((None, 32, 32, 3), tf.float32, name='input')]
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=input_sig, opset=13)

with open('mnist_cnn_32x32.onnx', 'wb') as f:
    f.write(model_proto.SerializeToString())

print('ONNX model saved.')

# Calibration data for STEdgeAI quantisation
np.savez('mnist_calib.npz', input_1=x_test[:200].astype('float32'))
print('Calibration file saved.')

In [ ]:
from google.colab import files
files.download('mnist_cnn_32x32.onnx')
files.download('mnist_calib.npz')

## Next steps — back on the PC

1. Run STEdgeAI conversion on `mnist_cnn_32x32.onnx`  
   → produces new `network.c`, `stai_network.c`, `stai_network.h`, `network_ecblobs.h`, `network_data.hex`

2. Copy those files into `Model/` (replace the old ones)

3. STM32CubeIDE → **Build All**

4. Flash:
   ```powershell
   .\flash.ps1 -Weights    # new weights into NOR flash
   .\flash.ps1 -AppOnly    # new firmware binary
   ```

No changes to `main.c` or any other firmware file are needed.